# Missing Values Exercises

## Gradescope Autograding

Please follow [all standard guidance](https://www.practicaldatascience.org/html/autograder_guidelines.html) for submitting this assignment to the Gradescope autograder, including storing your solutions in a dictionary called `results` and ensuring your notebook runs from the start to completion without any errors.

For this assignment, please name your file `exercise_missing.ipynb` before uploading.

You can check that you have answers for all questions in your `results` dictionary with this code:

assert set(results.keys()) == {
    "ex2_avg_income",
    "ex3_share_making_9999999",
    "ex3_share_making_zero",
    "ex5_avg_income",
    "ex8_avg_income_black",
    "ex8_avg_income_white",
    "ex8_racial_difference",
    "ex9_avg_income_black",
    "ex9_avg_income_white",
    "ex10_wage_gap",
}

### Submission Limits
Please remember that you are **only allowed three submissions to the autograder**. Your last submission (if you submit 3 or fewer times), or your third submission (if you submit more than 3 times) will determine your grade Submissions that error out will **not** count against this total.

## Exercises

### Exercise 1

Today, we will be using the ACS data we used during out first `pandas` exercise to examine the US income distribution, and how it varies by race. Note that because the US income distribution has a very small number of people with *extremely* high incomes, and the ACS is just a sample of Americans, the far right tail of the distribution will not be very well estimated. However, this data should suffice for helping to understand wealth inequality in the United States.

To begin, load the ACS Data we used in our first pandas exercise. That [data can be found here](https://github.com/nickeubank/MIDS_Data/tree/master/US_AmericanCommunitySurvey). We’ll be working with `US_ACS_2017_10pct_sample.dta`.

In [1]:
import pandas as pd
import numpy as np

In [2]:
acs = pd.read_stata(
    "https://github.com/nickeubank/MIDS_Data/raw/refs/heads/master/US_AmericanCommunitySurvey/US_ACS_2017_10pct_sample.dta"
)

results = {}

### Exercise 2
Let’s begin by calculating the mean US incomes from this data (recall that income is stored in the `inctot` variable). Store the answer in `results` under the key `"ex2_avg_income"`.

In [3]:
results["ex2_avg_income"] = acs["inctot"].mean()

print(f"The average income is: ${results['ex2_avg_income']:.2f}")

The average income is: $1723646.27


### Exercise 3
Hmmm… That doesn’t look right. The average American is definitely not earning that much a year! Let’s look at the values of `inctot` using `value_counts()`. Do you see a problem?

In [4]:
acs["inctot"].value_counts()

inctot
9999999    53901
0          33679
30000       4778
50000       4414
40000       4413
           ...  
41620          1
17510          1
74930          1
42130          1
505400         1
Name: count, Length: 8471, dtype: int64

> Yes, there is a problem. There are over 50,000 people with an income of $9,999,999. That's way too many millionaires!

Now use `value_counts()` with the argument `normalize=True` to see proportions of the sample that report each value instead of the count of people in each category. 

In [5]:
p_inctot = acs["inctot"].value_counts(normalize=True)
p_inctot

inctot
9999999    0.168967
0          0.105575
30000      0.014978
50000      0.013837
40000      0.013834
             ...   
41620      0.000003
17510      0.000003
74930      0.000003
42130      0.000003
505400     0.000003
Name: proportion, Length: 8471, dtype: float64

What percentage of our sample has an income of 9,999,999? Store that proportion (between 0 and 1) as `"ex3_share_making_9999999"`. What percentage has an income of 0? Store that proportion as `"ex3_share_making_zero"`.

(Recall `.value_counts()` returns a Series, so you can pull values out with our usual pandas tools.)

In [6]:
results["ex3_share_making_9999999"] = p_inctot[9999999]
results["ex3_share_making_zero"] = p_inctot[0]

> The proportion making $9,999,999 is 0.169 while the proportion making $0 is 0.106

### Exercise 4

As we discussed before, the ACS uses a value of 9999999 to denote that income information is not available for someone. The problem with using this kind of “sentinel value” is that pandas doesn’t understand that this is supposed to denote missing data, and so when it averages the variable, it doesn’t know to ignore 9999999.

To help out `pandas`, use the `replace` command to replace all values of 9999999 with `np.nan`.

In [7]:
# wasn't there an easier way to do this?
acs = acs.replace({"inctot": {9999999: np.nan}})
# acs['inctot'].value_counts()

### Exercise 5

Now that we’ve properly labeled our missing data as `np.nan`, let’s calculate the average US income once more. Store the answer in `results` under the key `"ex5_avg_income"`.

In [8]:
results["ex5_avg_income"] = acs["inctot"].mean()

print(f"The corrected average income is: ${results['ex5_avg_income']:.2f}")

The corrected average income is: $40890.18


> The value of $40,890.18 makes a lot more sense.

### Exercise 6

OK, now we’ve been able to get a reasonable average income number. As we can see, a major advantage of using `np.nan` is that `pandas` knows that `np.nan` observations should just be ignored when we are calculating means.

But it’s not enough to just get rid of the people who had `inctot` values of 9999999. We also need to know why those values were missing. Suppose, for example, that the value of 9999999 was used for anyone who made more than 100,000 dollars: if we just dropped those people, then our estimate of average income wouldn’t mean much, would it?

So let’s make sure we understand *why* data is missing for some people. If you recall from our last exercise, it seemed to be the case that most of the people who had incomes of 9999999 were children. Let’s make sure that’s true by looking at the distribution of the variable `age` for people for whom `inctot` is missing (i.e. subset the data to people with inctot missing, then look at the values of `age` with `value_counts()`).

In [9]:
inc_missing = acs[acs["inctot"].isna()]
inc_missing["age"].value_counts()

age
10    3997
9     3977
14    3847
12    3845
13    3800
      ... 
92       0
93       0
94       0
95       0
96       0
Name: count, Length: 97, dtype: int64

Then do the opposite: look at the distribution of the `age` variable for people who whom `inctot` is not missing.

Can you determine when 9999999 was being used? Is it ok we’re excluding those people from our analysis?

Note: In this data, Python doesn’t understand `age` is a number; it thinks it is a string because the original data has categories like “90 (90+ in 1980 and 1990)” and “less than 1 year old”. So you can’t just use `min()` or `max()`. We’ll discuss converting string variables into numbers in a future class.

In [10]:
inc_missing = acs[acs["inctot"].notnull()]
inc_missing["age"].value_counts()

age
60    4950
54    4821
56    4776
59    4776
58    4734
      ... 
10       0
11       0
12       0
13       0
14       0
Name: count, Length: 97, dtype: int64

> 9999999 was being used for children under the age of 15. Therefore, it is most likely fine to exclude them from the analysis because they are unlikely to be contributing to the household finances (they may be earning money, but it is more likely allowance).

### Exercise 7

Great, so now we know why those people had missing data, and we’re ok with excluding them.

But as we previously noted, there are also a lot of observations of zero income in our data, and it’s not clear that we want everyone with a zero-income *should* be included in this average, since those may be people who are retired, or in school.

Let’s limit our attention to people who are currently working by subsetting to only employed respondents. We can do this using `empstat`. Remember you can use `value_counts()` to see what values of empstat are in the data!

In [11]:
acs_employed = inc_missing[inc_missing["empstat"] == "employed"]
acs_employed

,year,datanum,serial,cbserial,numprec,subsamp,hhwt,hhtype,cluster,adjust,...,migcounty1,migmet131,vetdisab,diffrem,diffphys,diffmob,diffcare,diffsens,diffeye,diffhear
1,2017,1,1200045,2.017001e+12,6,79,25,"male householder, no wife present",2.017012e+12,1.011189,...,0,not in identifiable area,n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
2,2017,1,70831,2.017000e+12,1 person record,36,57,"male householder, living alone",2.017001e+12,1.011189,...,0,not in identifiable area,n/a,has cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
5,2017,1,563897,2.017001e+12,3,19,66,"female householder, no husband present",2.017006e+12,1.011189,...,0,not in identifiable area,n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
9,2017,1,856859,2.017001e+12,5,69,12,married-couple family household,2.017009e+12,1.011189,...,0,not in identifiable area,n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
10,2017,1,175930,2.017001e+12,9,72,171,married-couple family household,2.017002e+12,1.011189,...,0,not in identifiable area,n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318995,2017,1,46231,2.017001e+12,3,36,104,married-couple family household,2.017000e+12,1.011189,...,13,"phoenix-mesa-scottsdale, az",n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
318999,2017,1,734396,2.017001e+12,4,78,100,married-couple family household,2.017007e+12,1.011189,...,0,not in identifiable area,n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
319001,2017,1,510444,2.017001e+12,2,43,152,"female householder, no husband present",2.017005e+12,1.011189,...,0,not in identifiable area,n/a,has cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no
319002,2017,1,1220474,2.017001e+12,4,16,148,married-couple family household,2.017012e+12,1.011189,...,0,not in identifiable area,n/a,no cognitive difficulty,no ambulatory difficulty,no independent living difficulty,no,no vision or hearing difficulty,no,no


### Exercise 8

Now let’s estimate the racial income gap in the United States. What is the average salary for employed Black Americans, and what is the average salary for employed White Americans? In percentage terms, how much **more** does the average White American make than the average Black American (i.e., a value of `100` could imply the average White American makes 2x what a Black American makes)?

**Note:** these values are not quite accurate estimates. As we’ll discuss in later lessons, to get completely accurate estimates from the ACS we have to take into account how people were selected to be interviewed. But you get pretty good estimates in most cases even without weights—your estimate of the racial wage gap without weights is within 5% of the corrected value.

**Note:** This is actually an underestimate of the wage gap. The US Census treats Hispanic respondents as a sub-category of “White.” While all ethnic distinctions are socially constructed, and so on some level these distinctions are all deeply problematic, this coding is inconsistent with what most Americans think of when they hear the term “White,” a term *most* Americans think of as a category that is mutually exclusive of being Hispanic or Latino (categories which are also usually conflated in American popular discussion). With that in mind, most researchers working with US Census data split “White” into “White, Hispanic” and “White, Non-Hispanic” using `race` *and* `hispan`. But for the moment, just identify “White” respondents using the value in `race`.

Store your results in `results` under the keys `"ex8_avg_income_black"`, `"ex8_avg_income_white"`, and the percentage difference as `ex8_racial_difference`. Please note the wording above when calculating the percentage difference to ensure you get the reference category correct, and interpret your result as well.

In [12]:
# ex8_avg_income_black - average salary of employed black americans

results["ex8_avg_income_black"] = acs_employed.loc[
    acs_employed["race"] == "black/african american/negro", "inctot"
].mean()

# ex8_avg_income_white - average salary of employed white americans

results["ex8_avg_income_white"] = acs_employed.loc[
    acs_employed["race"] == "white", "inctot"
].mean()

# ex8_racial_difference - difference between the two averages above as a percentage
results["ex8_racial_difference"] = (
    (results["ex8_avg_income_white"] - results["ex8_avg_income_black"])
    / results["ex8_avg_income_black"]
) * 100
print(
    f"The average income for black Americans is ${results['ex8_avg_income_black']:.2f}"
)
print(
    f"The average income for white Americans is ${results['ex8_avg_income_white']:.2f}"
)
print(f"The racial income difference is {results['ex8_racial_difference']:.2f} %")

The average income for black Americans is $41747.95
The average income for white Americans is $60473.15
The racial income difference is 44.85 %


> This implies that white Americans make almost 50 % more than black Americans

### Exercise 9

As noted above, these estimates are not actually *quite* correct because we aren’t taking into account the fact that when the US Census decided who to survey, not all Americans had the same likelihood of being asked. The US American Community Survey is an example of a *weighted* survey (essentially, people from smaller subpopulations have a higher likelihood of being included to ensure enough individuals in the final survey to constitute a representative sample that can be used statistically).

To calculate a weighted average that takes into account these survey weights (in other words, a more accurate estimate of US incomes), you need to use the following formula:

$$weighted\_mean\_of\_x = \frac{\sum_i x_i * weight_i}{\sum_i weight_i}$$
 
(As you can see, when $weight_i$ is constant for all observations, this just simplifies to our normal formula for mean values. It is only when weights vary across individuals that weights must be explicitly addressed).

In this data, weights are stored in the variable `perwt`, which is the number of people for which each observation is a stand-in (the inverse of that observation’s sampling probability).

Using the formula, re-calculate the *weighted* average income for both populations and store them as `ex9_avg_income_white` and `ex9_avg_income_black`.

In [13]:
def weighted_mean(df, x, weights):
    """
    df is the data frame with the corresponding columns for x and weights
    x is the column name for the variable of interest
    weights is the column name for the weight assigned to the observations
    """
    return (df[x] * df[weights]).sum() / df[weights].sum()


# ex9_avg_income_white - weighted average salary of employed white americans
results["ex9_avg_income_white"] = weighted_mean(
    acs_employed.loc[acs_employed["race"] == "white"], "inctot", "perwt"
)
print(
    f"The weighted average income for white Americans is ${results['ex9_avg_income_white']:.2f}"
)

# ex9_avg_income_black - weighted average salary of employed black americans
results["ex9_avg_income_black"] = weighted_mean(
    acs_employed.loc[acs_employed["race"] == "black/african american/negro"],
    "inctot",
    "perwt",
)
print(
    f"The weighted average income for black Americans is ${results['ex9_avg_income_black']:.2f}"
)

The weighted average income for white Americans is $58361.48
The weighted average income for black Americans is $40430.95


### Exercise 10

Now calculate the **weighted** average income gap between *non-Hispanic* White Americans and Black Americans. What percentage more do employed White non-Hispanic Americans earn than employed Black Americans? Store as `"ex10_wage_gap"`.

In [14]:
acs_employed["hispan"].value_counts()

hispan
not hispanic    128293
mexican          12437
other             5302
puerto rican      1787
cuban              939
Name: count, dtype: int64

In [15]:
# weighted average income for "not hispanic" white americans
non_hisp_white_income = weighted_mean(
    acs_employed.loc[
        (acs_employed["race"] == "white") & (acs_employed["hispan"] == "not hispanic")
    ],
    "inctot",
    "perwt",
)
print(
    f"The weighted average income for non-Hispanic white Americans is ${non_hisp_white_income:.2f}"
)

# ex10_wage_gap - difference between the non-hisp-white and black incomes as a percentage
results["ex10_wage_gap"] = (
    (non_hisp_white_income - results["ex9_avg_income_black"])
    / results["ex9_avg_income_black"]
) * 100
print(f"The racial income difference is {results['ex10_wage_gap']:.2f}%")

The weighted average income for non-Hispanic white Americans is $61669.29
The racial income difference is 52.53%


Exercise 11
Is that greater or less than the difference you found in Exercise 8? Why do you think that’s the case?

> This difference is more than what was found in Exercise 8. This is likely the case because non-Hispanic white Americans are more likely to be able to receive a better education, which is likely to lead to higher pay. There are other factors, but that would take a long time to discuss.

In [16]:
assert set(results.keys()) == {
    "ex2_avg_income",
    "ex3_share_making_9999999",
    "ex3_share_making_zero",
    "ex5_avg_income",
    "ex8_avg_income_black",
    "ex8_avg_income_white",
    "ex8_racial_difference",
    "ex9_avg_income_black",
    "ex9_avg_income_white",
    "ex10_wage_gap",
}

In [ ]:
"test"

'test'